In [1]:
import os
import pickle
import pandas as pd
import numpy as np

# Path to the metadata folder
metadata_dir = r"c:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\results\results_data\distnet_table4\metadata"

# Load all pkl files
all_results = []
for fname in os.listdir(metadata_dir):
    if fname.endswith('.pkl'):
        fpath = os.path.join(metadata_dir, fname)
        with open(fpath, 'rb') as f:
            data = pickle.load(f)
        all_results.append(data)

print(f"Total .pkl files loaded: {len(all_results)}")

# Preview keys
print("Keys in first result:", list(all_results[0].keys()))
print("Sample entry:")
for k, v in all_results[0].items():
    if k not in ('model_config', 'test_preds'):
        print(f"  {k}: {v}")

Total .pkl files loaded: 140
Keys in first result: ['model_name', 'model_config', 'scenario', 'fold', 'seed_context', 'seed_features', 'seed_samples_per_instance', 'feature_drop_rate', 'context_size', 'target_scale', 'subsample_method', 'num_samples_per_instance', 'use_cpu', 'early_stopping', 'early_stopping_patience', 'E_final', 'save_dir', 'n_epochs', 'batch_size', 'wc_time_limit', 'test_preds', 'random_state', 'result_metrics']
Sample entry:
  model_name: distnet
  scenario: clasp_factoring
  fold: 0
  seed_context: None
  seed_features: None
  seed_samples_per_instance: None
  feature_drop_rate: None
  context_size: None
  target_scale: max
  subsample_method: None
  num_samples_per_instance: 100
  use_cpu: True
  early_stopping: False
  early_stopping_patience: 50
  E_final: None
  save_dir: distnet_results/distnet_table4/
  n_epochs: 1000
  batch_size: 16
  wc_time_limit: 3540
  random_state: 0
  result_metrics: {'nllh': np.float64(-0.17769761669825596), 'fit_time': 3541.88361835

In [2]:
import pandas as pd
import numpy as np

# Build a flat dataframe
records = []
for r in all_results:
    records.append({
        'scenario':         r['scenario'],
        'fold':             r['fold'],
        'early_stopping':   r['early_stopping'],
        'nllh':             r['result_metrics']['nllh'],
    })

df = pd.DataFrame(records)
print("Shape:", df.shape)
print("\nUnique scenarios:", sorted(df['scenario'].unique()))
print("Unique early_stopping values:", df['early_stopping'].unique())
print("\nValue counts per (early_stopping, scenario, fold):")
print(df.groupby(['early_stopping', 'scenario', 'fold']).size().reset_index(name='count'))

Shape: (140, 4)

Unique scenarios: ['clasp_factoring', 'lpg-zeno', 'saps-CVVAR', 'spear_qcp', 'spear_swgcp', 'yalsat_qcp', 'yalsat_swgcp']
Unique early_stopping values: [False  True]

Value counts per (early_stopping, scenario, fold):
     early_stopping         scenario  fold  count
0             False  clasp_factoring     0      1
1             False  clasp_factoring     1      1
2             False  clasp_factoring     2      1
3             False  clasp_factoring     3      1
4             False  clasp_factoring     4      1
..              ...              ...   ...    ...
135            True     yalsat_swgcp     5      1
136            True     yalsat_swgcp     6      1
137            True     yalsat_swgcp     7      1
138            True     yalsat_swgcp     8      1
139            True     yalsat_swgcp     9      1

[140 rows x 4 columns]


In [3]:
# ── Assertions: exactly 10 folds per (early_stopping, scenario) ─────────────
print("=== Fold-count assertions ===")
fold_counts = (
    df.groupby(['early_stopping', 'scenario'])['fold']
    .nunique()
    .reset_index(name='n_folds')
)
print(fold_counts.to_string(index=False))

for _, row in fold_counts.iterrows():
    assert row['n_folds'] == 10, (
        f"Expected 10 folds for early_stopping={row['early_stopping']}, "
        f"scenario={row['scenario']}, but got {row['n_folds']}"
    )
print("\n✅  All scenarios have exactly 10 folds for each early-stopping setting.")

=== Fold-count assertions ===
 early_stopping        scenario  n_folds
          False clasp_factoring       10
          False        lpg-zeno       10
          False      saps-CVVAR       10
          False       spear_qcp       10
          False     spear_swgcp       10
          False      yalsat_qcp       10
          False    yalsat_swgcp       10
           True clasp_factoring       10
           True        lpg-zeno       10
           True      saps-CVVAR       10
           True       spear_qcp       10
           True     spear_swgcp       10
           True      yalsat_qcp       10
           True    yalsat_swgcp       10

✅  All scenarios have exactly 10 folds for each early-stopping setting.


In [4]:
# ── Aggregate: mean NLLH over 10 folds ──────────────────────────────────────
summary = (
    df.groupby(['early_stopping', 'scenario'])['nllh']
    .agg(mean_nllh='mean', std_nllh='std')
    .reset_index()
)

# ── Table 1: WITH early stopping ─────────────────────────────────────────────
es_true = (
    summary[summary['early_stopping'] == True]
    .drop(columns='early_stopping')
    .sort_values('scenario')
    .reset_index(drop=True)
)
es_true.columns = ['Scenario', 'Mean NLLH (10-fold)', 'Std NLLH']
es_true['Mean NLLH (10-fold)'] = es_true['Mean NLLH (10-fold)'].round(4)
es_true['Std NLLH']            = es_true['Std NLLH'].round(4)

print("=" * 55)
print("  Table 1 – DistNet  |  WITH Early Stopping")
print("=" * 55)
print(es_true.to_string(index=False))

# ── Table 2: WITHOUT early stopping ──────────────────────────────────────────
es_false = (
    summary[summary['early_stopping'] == False]
    .drop(columns='early_stopping')
    .sort_values('scenario')
    .reset_index(drop=True)
)
es_false.columns = ['Scenario', 'Mean NLLH (10-fold)', 'Std NLLH']
es_false['Mean NLLH (10-fold)'] = es_false['Mean NLLH (10-fold)'].round(4)
es_false['Std NLLH']            = es_false['Std NLLH'].round(4)

print("\n" + "=" * 55)
print("  Table 2 – DistNet  |  WITHOUT Early Stopping")
print("=" * 55)
print(es_false.to_string(index=False))

  Table 1 – DistNet  |  WITH Early Stopping
       Scenario  Mean NLLH (10-fold)  Std NLLH
clasp_factoring              -0.1480    0.0223
       lpg-zeno              -0.8526    0.0094
     saps-CVVAR              -0.6320    0.0287
      spear_qcp              -1.1044    0.0199
    spear_swgcp              -0.4731    0.0129
     yalsat_qcp              -0.7555    0.0077
   yalsat_swgcp              -0.8062    0.0108

  Table 2 – DistNet  |  WITHOUT Early Stopping
       Scenario  Mean NLLH (10-fold)  Std NLLH
clasp_factoring              -0.1533    0.0252
       lpg-zeno              -0.8497    0.0079
     saps-CVVAR              -0.5687    0.0747
      spear_qcp              -1.0988    0.0218
    spear_swgcp              -0.4694    0.0121
     yalsat_qcp              -0.7498    0.0111
   yalsat_swgcp              -0.8046    0.0112


In [5]:
# ── Aggregate: mean NLLH over 10 folds ──────────────────────────────────────
summary = (
    df.groupby(['early_stopping', 'scenario'])['nllh']
    .agg(mean_nllh='mean', std_nllh='std')
    .reset_index()
)

# ── Table 1: WITH early stopping ─────────────────────────────────────────────
es_true = (
    summary[summary['early_stopping'] == True]
    .drop(columns='early_stopping')
    .sort_values('scenario')
    .reset_index(drop=True)
)
es_true.columns = ['Scenario', 'Mean NLLH (10-fold)', 'Std NLLH']
es_true['Mean NLLH (10-fold)'] = es_true['Mean NLLH (10-fold)'].round(4)
es_true['Std NLLH']            = es_true['Std NLLH'].round(4)

print("=" * 55)
print("  Table 1 – DistNet  |  WITH Early Stopping")
print("=" * 55)
print(es_true.to_string(index=False))

# ── Table 2: WITHOUT early stopping ──────────────────────────────────────────
es_false = (
    summary[summary['early_stopping'] == False]
    .drop(columns='early_stopping')
    .sort_values('scenario')
    .reset_index(drop=True)
)
es_false.columns = ['Scenario', 'Mean NLLH (10-fold)', 'Std NLLH']
es_false['Mean NLLH (10-fold)'] = es_false['Mean NLLH (10-fold)'].round(4)
es_false['Std NLLH']            = es_false['Std NLLH'].round(4)

print("\n" + "=" * 55)
print("  Table 2 – DistNet  |  WITHOUT Early Stopping")
print("=" * 55)
print(es_false.to_string(index=False))

# ── Save to CSV ───────────────────────────────────────────────────────────────
output_dir = r"c:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\results\results_data\distnet_table4"

es_true_path  = os.path.join(output_dir, "distnet_results_with_early_stopping.csv")
es_false_path = os.path.join(output_dir, "distnet_results_without_early_stopping.csv")

es_true.to_csv(es_true_path,  index=False)
es_false.to_csv(es_false_path, index=False)

print(f"\n✅  Saved: {es_true_path}")
print(f"✅  Saved: {es_false_path}")

  Table 1 – DistNet  |  WITH Early Stopping
       Scenario  Mean NLLH (10-fold)  Std NLLH
clasp_factoring              -0.1480    0.0223
       lpg-zeno              -0.8526    0.0094
     saps-CVVAR              -0.6320    0.0287
      spear_qcp              -1.1044    0.0199
    spear_swgcp              -0.4731    0.0129
     yalsat_qcp              -0.7555    0.0077
   yalsat_swgcp              -0.8062    0.0108

  Table 2 – DistNet  |  WITHOUT Early Stopping
       Scenario  Mean NLLH (10-fold)  Std NLLH
clasp_factoring              -0.1533    0.0252
       lpg-zeno              -0.8497    0.0079
     saps-CVVAR              -0.5687    0.0747
      spear_qcp              -1.0988    0.0218
    spear_swgcp              -0.4694    0.0121
     yalsat_qcp              -0.7498    0.0111
   yalsat_swgcp              -0.8046    0.0112

✅  Saved: c:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\results\results_data\distnet_table4\distnet_results_with_early_stopping.csv
✅  Saved: